In [1]:
print("=" * 60)
print("Site Readiness Index (SRI) - Suitability Model")
print('City of Red Wing Land Development Analysis')
print('GIS Subconsultant: Geospatial Business Intelligence LLC')
print(' ')
print('Dimensions & Weights:'
    '\nPhysical & Infrastructure  25%'
    '\nRegulatory                 20%'
    '\nMarket                     20%'
    '\nOwnership                  20%'
    '\nStrategic                  15%')
print("=" * 60)

Site Readiness Index (SRI) - Suitability Model
City of Red Wing Land Development Analysis
GIS Subconsultant: Geospatial Business Intelligence LLC
 
Dimensions & Weights:
Physical & Infrastructure  25%
Regulatory                 20%
Market                     20%
Ownership                  20%
Strategic                  15%


### Import Libraries

In [2]:
import pandas as pd 
import arcpy
import os 
import glob 
import numpy as np
import h3

### Set Environments

In [3]:
workingfolder = r'C:\Users\Thepr\Documents\Consulting\CityofRedWing'
projectGDB = os.path.join(workingfolder, 'ArcGISPro', 'RedWing.gdb')
outputGDB = os.path.join(workingfolder, 'ArcGISPro', 'RedWingOutput.gdb')


arcpy.env.workspace = projectGDB
arcpy.env.overwriteOutput = True

spatialreference = arcpy.SpatialReference(26859)

arcpy.env.outputCoordinateSystem = spatialreference

print(f'The arcpy version being use is {arcpy.GetInstallInfo()["Version"]}')

The arcpy version being use is 3.2


## Parcel

In [4]:
parcel = os.path.join(projectGDB,'Site')

In [5]:
print([row.name for row in arcpy.ListFields(parcel)])

['OBJECTID', 'Shape', 'Shape_Leng', 'ID', 'Name', 'Total_acre', 'Acres_chal', 'Acres_cons', 'Acres_deve', 'Percent_de', 'Other', 'Acres_TOC_', 'Acres_TOC1', 'Acres_wetl', 'Acres_floo', 'Acres_high', 'Acres_CE', 'Utilities', 'Category', 'MERGE_SRC', 'Shape_Length', 'Shape_Area']


In [6]:
arcpy.management.RepairGeometry(parcel)

<Result 'C:\\Users\\Thepr\\Documents\\Consulting\\CityofRedWing\\ArcGISPro\\RedWing.gdb\\Site'>

## Zoning 

- Join Zoning to Parcel

In [7]:
zoning = os.path.join(projectGDB,'ZoningRW')
print("Input FC:", arcpy.Exists(zoning))

Input FC: True


In [8]:
arcpy.management.RepairGeometry(zoning)

<Result 'C:\\Users\\Thepr\\Documents\\Consulting\\CityofRedWing\\ArcGISPro\\RedWing.gdb\\ZoningRW'>

In [9]:
parcelzoning = os.path.join(outputGDB,'ParcelZoning')

fieldmapping = arcpy.FieldMappings()
fieldmapping.addTable(parcel)
fieldmapping.addTable(zoning)

analysisfields = ['PIN','C0CITY','ZONE_DOC','ID', 'Name']

for field in fieldmapping.fields:
    if field.name not in analysisfields:
        fieldmapping.removeFieldMap(fieldmapping.findFieldMapIndex(field.name))

In [10]:
if arcpy.Exists(parcelzoning):
    arcpy.Delete_management(parcelzoning)

arcpy.analysis.SpatialJoin(
    target_features= parcel,
    join_features=zoning,
    out_feature_class=parcelzoning ,
    join_operation="JOIN_ONE_TO_ONE",
    join_type="KEEP_ALL",
    field_mapping=fieldmapping,
    match_option="LARGEST_OVERLAP",
    search_radius=None,
    distance_field_name="",
    match_fields=None)

<Result 'C:\\Users\\Thepr\\Documents\\Consulting\\CityofRedWing\\ArcGISPro\\RedWingOutput.gdb\\ParcelZoning'>

## Future Land Use

- Join future landuse land to the Parcel and Zoning Joined Layer

In [11]:
futurelu = os.path.join(projectGDB,'FutureLU')
print("Input FC:", arcpy.Exists(futurelu))

Input FC: True


In [12]:
arcpy.management.RepairGeometry(futurelu)

<Result 'C:\\Users\\Thepr\\Documents\\Consulting\\CityofRedWing\\ArcGISPro\\RedWing.gdb\\FutureLU'>

In [13]:
parcelzoningFLU = os.path.join(outputGDB,'ParcelZoningFLU')

fieldmapping = arcpy.FieldMappings()
fieldmapping.addTable(parcelzoning)
fieldmapping.addTable(futurelu)

analysisfields = ['PIN','C0CITY','ZONE_DOC','LANDUSEDES','DEV_STATUS','C0CLSFDS','C0CITY','C0CITYP',
                  'C0PYEAR','LANDUSEDES','PLU_1','C0HSTDDS', 'REDEV','ID', 'Name']

for field in fieldmapping.fields:
    if field.name not in analysisfields:
        fieldmapping.removeFieldMap(fieldmapping.findFieldMapIndex(field.name))

In [14]:
if arcpy.Exists(parcelzoningFLU):
    arcpy.Delete_management(parcelzoningFLU)

arcpy.analysis.SpatialJoin(
    target_features= parcelzoning,
    join_features=futurelu,
    out_feature_class=parcelzoningFLU ,
    join_operation="JOIN_ONE_TO_ONE",
    join_type="KEEP_ALL",
    field_mapping=fieldmapping,
    match_option="LARGEST_OVERLAP",
    search_radius=None,
    distance_field_name="",
    match_fields=None)

<Result 'C:\\Users\\Thepr\\Documents\\Consulting\\CityofRedWing\\ArcGISPro\\RedWingOutput.gdb\\ParcelZoningFLU'>

## Conservation Easement

In [15]:
conservation = os.path.join(projectGDB,'ConservationEasementsRW')
arcpy.management.RepairGeometry(conservation)

parcelzoningFLU = os.path.join(outputGDB,'ParcelZoningFLU')

- Intersect Conservation Easement with Parcels

In [16]:
fields = [f.name for f in arcpy.ListFields(parcelzoningFLU)]

if "ConservationEasement" not in fields:
    arcpy.management.AddField(parcelzoningFLU, "ConservationEasement", "TEXT", field_length=10)

intersectingids = set()

intersectparcels = arcpy.management.SelectLayerByLocation(parcelzoningFLU,"INTERSECT", conservation)

with arcpy.da.SearchCursor(intersectparcels, ["OID@"]) as cursor:
    for row in cursor:
        intersectingids.add(row[0])

print(f"{len(intersectingids)} parcels intersect conservation layer.")

with arcpy.da.UpdateCursor(parcelzoningFLU, ["OID@", "ConservationEasement"]) as cursor:
    for row in cursor:
        if row[0] in intersectingids:
            row[1] = "Yes"
        else:
            row[1] = "No"
        cursor.updateRow(row)

print("Done. ConservationEasement field updated on parcels.")

3 parcels intersect conservation layer.
Done. ConservationEasement field updated on parcels.


## Tribal Areas of Concern

In [17]:
tribalareasofconcern = os.path.join(projectGDB,'TribalAreasOfConcern')

arcpy.management.RepairGeometry(tribalareasofconcern)

parcelzoningFLU = os.path.join(outputGDB,'ParcelZoningFLU')

In [18]:
fields = [f.name for f in arcpy.ListFields(parcelzoningFLU)]

if "TribalAreas" not in fields:
    arcpy.management.AddField(parcelzoningFLU, "TribalAreas", "TEXT", field_length=10)

intersectingids = set()

intersectparcels = arcpy.management.SelectLayerByLocation(parcelzoningFLU,"INTERSECT", tribalareasofconcern)

with arcpy.da.SearchCursor(intersectparcels, ["OID@"]) as cursor:
    for row in cursor:
        intersectingids.add(row[0])

print(f"{len(intersectingids)} parcels intersect TribalAreas layer.")

with arcpy.da.UpdateCursor(parcelzoningFLU, ["OID@", "TribalAreas"]) as cursor:
    for row in cursor:
        if row[0] in intersectingids:
            row[1] = "Yes"
        else:
            row[1] = "No"
        cursor.updateRow(row)

print("Done. tribalareasofconcern field updated on parcels.")

6 parcels intersect TribalAreas layer.
Done. tribalareasofconcern field updated on parcels.


## SE Historical

In [19]:
sehistorical = os.path.join(projectGDB,'SEHistDistLocal')

arcpy.management.RepairGeometry(sehistorical)

parcelzoningFLU = os.path.join(outputGDB,'ParcelZoningFLU')

In [20]:
fields = [f.name for f in arcpy.ListFields(parcelzoningFLU)]

if "SEHistDistLocal" not in fields:
    arcpy.management.AddField(parcelzoningFLU, "SEHistDistLocal", "TEXT", field_length=10)

intersectingids = set()

intersectparcels = arcpy.management.SelectLayerByLocation(parcelzoningFLU,"INTERSECT", sehistorical)

with arcpy.da.SearchCursor(intersectparcels, ["OID@"]) as cursor:
    for row in cursor:
        intersectingids.add(row[0])

print(f"{len(intersectingids)} parcels intersect HistDistLocal layer.")

with arcpy.da.UpdateCursor(parcelzoningFLU, ["OID@", "SEHistDistLocal"]) as cursor:
    for row in cursor:
        if row[0] in intersectingids:
            row[1] = "Yes"
        else:
            row[1] = "No"
        cursor.updateRow(row)

print("Done. SE Hist District field updated on parcels.")

0 parcels intersect HistDistLocal layer.
Done. SE Hist District field updated on parcels.


## Historial Local

In [21]:
histdistdocal = os.path.join(projectGDB, 'HistDistLocal')

arcpy.management.RepairGeometry(histdistdocal)

print("Input FC:", arcpy.Exists(histdistdocal))

Input FC: True


In [22]:
fields = [f.name for f in arcpy.ListFields(parcelzoningFLU)]

if "HistDistLocal" not in fields:
    arcpy.management.AddField(parcelzoningFLU, "HistDistLocal", "TEXT", field_length=10)

intersectingids = set()

intersectparcels = arcpy.management.SelectLayerByLocation(parcelzoningFLU,"INTERSECT", histdistdocal)

with arcpy.da.SearchCursor(intersectparcels, ["OID@"]) as cursor:
    for row in cursor:
        intersectingids.add(row[0])

print(f"{len(intersectingids)} parcels intersect HistDistLocal layer.")

with arcpy.da.UpdateCursor(parcelzoningFLU, ["OID@", "HistDistLocal"]) as cursor:
    for row in cursor:
        if row[0] in intersectingids:
            row[1] = "Yes"
        else:
            row[1] = "No"
        cursor.updateRow(row)

print("Done. HistDistLocal District field updated on parcels.")

1 parcels intersect HistDistLocal layer.
Done. HistDistLocal District field updated on parcels.


### Green network

In [23]:
greennetwork = os.path.join(projectGDB,'GreenNetworkRW')

arcpy.management.RepairGeometry(greennetwork)

print("Input FC:", arcpy.Exists(greennetwork))

Input FC: True


In [24]:
fields = [f.name for f in arcpy.ListFields(parcelzoningFLU)]

if "GreenNetwork" not in fields:
    arcpy.management.AddField(parcelzoningFLU, "GreenNetwork", "TEXT", field_length=10)

intersectingids = set()

intersectparcels = arcpy.management.SelectLayerByLocation(parcelzoningFLU,"INTERSECT", greennetwork)

with arcpy.da.SearchCursor(intersectparcels, ["OID@"]) as cursor:
    for row in cursor:
        intersectingids.add(row[0])

print(f"{len(intersectingids)} parcels intersect greennetwork layer.")

with arcpy.da.UpdateCursor(parcelzoningFLU, ["OID@", "GreenNetwork"]) as cursor:
    for row in cursor:
        if row[0] in intersectingids:
            row[1] = "Yes"
        else:
            row[1] = "No"
        cursor.updateRow(row)

print("Done. greennetwork  field updated on parcels.")

13 parcels intersect greennetwork layer.
Done. greennetwork  field updated on parcels.


### PICC Owned

In [25]:
piccowned = os.path.join(projectGDB,'PIIC_Owned')

arcpy.management.RepairGeometry(piccowned)

parcelzoningFLU = os.path.join(outputGDB,'ParcelZoningFLU')

In [26]:
fields = [f.name for f in arcpy.ListFields(parcelzoningFLU)]

if "PIICOwned" not in fields:
    arcpy.management.AddField(parcelzoningFLU, "PIICOwned", "TEXT", field_length=10)

intersectingids = set()

intersectparcels = arcpy.management.SelectLayerByLocation(parcelzoningFLU,"INTERSECT", piccowned)

with arcpy.da.SearchCursor(intersectparcels, ["OID@"]) as cursor:
    for row in cursor:
        intersectingids.add(row[0])

print(f"{len(intersectingids)} parcels intersect PIICOwned layer.")

with arcpy.da.UpdateCursor(parcelzoningFLU, ["OID@", "PIICOwned"]) as cursor:
    for row in cursor:
        if row[0] in intersectingids:
            row[1] = "Yes"
        else:
            row[1] = "No"
        cursor.updateRow(row)

print("Done. PIICOwned  field updated on parcels.")

1 parcels intersect PIICOwned layer.
Done. PIICOwned  field updated on parcels.


### Locality 

In [27]:
rwlocality = os.path.join(projectGDB,'RW_Locality')

arcpy.management.RepairGeometry(rwlocality)

print("Input FC:", arcpy.Exists(rwlocality))

Input FC: True


In [28]:
fields = [f.name for f in arcpy.ListFields(parcelzoningFLU)]

if "RWLocality" not in fields:
    arcpy.management.AddField(parcelzoningFLU, "RWLocality", "TEXT", field_length=10)

intersectingids = set()

intersectparcels = arcpy.management.SelectLayerByLocation(parcelzoningFLU,"INTERSECT", rwlocality)

with arcpy.da.SearchCursor(intersectparcels, ["OID@"]) as cursor:
    for row in cursor:
        intersectingids.add(row[0])

print(f"{len(intersectingids)} parcels intersect PIICOwned layer.")

with arcpy.da.UpdateCursor(parcelzoningFLU, ["OID@", "RWLocality"]) as cursor:
    for row in cursor:
        if row[0] in intersectingids:
            row[1] = "Yes"
        else:
            row[1] = "No"
        cursor.updateRow(row)

print("Done. RWLocality  field updated on parcels.")

5 parcels intersect PIICOwned layer.
Done. RWLocality  field updated on parcels.


### Water Main

In [29]:
watermains = os.path.join(projectGDB,'WaterMainsRW')

arcpy.management.RepairGeometry(watermains)

parcelzoningFLU = os.path.join(outputGDB,'ParcelZoningFLU')

print("Input FC:", arcpy.Exists(watermains))

Input FC: True


In [30]:
arcpy.analysis.Near(
    in_features=parcelzoningFLU,
    near_features=watermains,
    search_radius=None,
    location="NO_LOCATION",
    angle="NO_ANGLE",
    method="PLANAR",
    field_names="NEAR_DIST NEAR_DIST",
    distance_unit="Feet")

<Result 'C:\\Users\\Thepr\\Documents\\Consulting\\CityofRedWing\\ArcGISPro\\RedWingOutput.gdb\\ParcelZoningFLU'>

In [31]:
existingfields = [row.name for row in arcpy.ListFields(parcelzoningFLU)]
print(existingfields)

['OBJECTID', 'Shape', 'Join_Count', 'TARGET_FID', 'ID', 'Name', 'PIN', 'ZONE_DOC', 'C0CITY', 'C0CITYP', 'C0PYEAR', 'C0CLSFDS', 'C0HSTDDS', 'PLU_1', 'LANDUSEDES', 'REDEV', 'DEV_STATUS', 'Shape_Length', 'Shape_Area', 'ConservationEasement', 'TribalAreas', 'SEHistDistLocal', 'HistDistLocal', 'GreenNetwork', 'PIICOwned', 'RWLocality', 'NEAR_DIST', 'NEAR_FID']


In [32]:
arcpy.management.AlterField(
    in_table=parcelzoningFLU,
    field="NEAR_DIST",
    new_field_name="DIST_WATERMAIN",
    new_field_alias="DIST_WATERMAIN",
    field_type="DOUBLE",
    field_length=20,
    field_is_nullable="NULLABLE")


<Result 'C:\\Users\\Thepr\\Documents\\Consulting\\CityofRedWing\\ArcGISPro\\RedWingOutput.gdb\\ParcelZoningFLU'>

In [33]:
arcpy.management.DeleteField(in_table=parcelzoningFLU,drop_field="NEAR_FID",method="DELETE_FIELDS")

<Result 'C:\\Users\\Thepr\\Documents\\Consulting\\CityofRedWing\\ArcGISPro\\RedWingOutput.gdb\\ParcelZoningFLU'>

### Sewer 

In [34]:
sewer = os.path.join(projectGDB,'SewerMainsRW')

arcpy.management.RepairGeometry(sewer)

parcelzoningFLU = os.path.join(outputGDB,'ParcelZoningFLU')

print("Input FC:", arcpy.Exists(sewer))

Input FC: True


In [35]:
arcpy.analysis.Near(
    in_features=parcelzoningFLU,
    near_features=sewer,
    search_radius=None,
    location="NO_LOCATION",
    angle="NO_ANGLE",
    method="PLANAR",
    field_names="NEAR_DIST NEAR_DIST",
    distance_unit="Feet")

<Result 'C:\\Users\\Thepr\\Documents\\Consulting\\CityofRedWing\\ArcGISPro\\RedWingOutput.gdb\\ParcelZoningFLU'>

In [36]:
arcpy.management.AlterField(
    in_table=parcelzoningFLU,
    field="NEAR_DIST",
    new_field_name="DIST_SEWER",
    new_field_alias="DIST_SEWER",
    field_type="DOUBLE",
    field_length=20,
    field_is_nullable="NULLABLE")

<Result 'C:\\Users\\Thepr\\Documents\\Consulting\\CityofRedWing\\ArcGISPro\\RedWingOutput.gdb\\ParcelZoningFLU'>

In [37]:
arcpy.management.DeleteField(in_table=parcelzoningFLU,drop_field="NEAR_FID",method="DELETE_FIELDS")

<Result 'C:\\Users\\Thepr\\Documents\\Consulting\\CityofRedWing\\ArcGISPro\\RedWingOutput.gdb\\ParcelZoningFLU'>

### Storm Pipes

In [38]:
stormpipes = os.path.join(projectGDB,'StormPipes')

arcpy.management.RepairGeometry(stormpipes)

parcelzoningFLU = os.path.join(outputGDB,'ParcelZoningFLU')

print("Input FC:", arcpy.Exists(stormpipes))

Input FC: True


In [39]:
arcpy.analysis.Near(
    in_features=parcelzoningFLU,
    near_features=stormpipes,
    search_radius=None,
    location="NO_LOCATION",
    angle="NO_ANGLE",
    method="PLANAR",
    field_names="NEAR_DIST NEAR_DIST",
    distance_unit="Feet")

<Result 'C:\\Users\\Thepr\\Documents\\Consulting\\CityofRedWing\\ArcGISPro\\RedWingOutput.gdb\\ParcelZoningFLU'>

In [40]:
arcpy.management.AlterField(
    in_table=parcelzoningFLU,
    field="NEAR_DIST",
    new_field_name="DIST_STORMPIPE",
    new_field_alias="DIST_STORMPIPE",
    field_type="DOUBLE",
    field_length=20,
    field_is_nullable="NULLABLE")

<Result 'C:\\Users\\Thepr\\Documents\\Consulting\\CityofRedWing\\ArcGISPro\\RedWingOutput.gdb\\ParcelZoningFLU'>

In [41]:
arcpy.management.DeleteField(in_table=parcelzoningFLU,drop_field="NEAR_FID",method="DELETE_FIELDS")

<Result 'C:\\Users\\Thepr\\Documents\\Consulting\\CityofRedWing\\ArcGISPro\\RedWingOutput.gdb\\ParcelZoningFLU'>

### Centerline RW 

In [42]:
centerline = os.path.join(projectGDB,'CenterlinesRW_PlusSect')

arcpy.management.RepairGeometry(centerline)

parcelzoningFLU = os.path.join(outputGDB,'ParcelZoningFLU')

print("Input FC:", arcpy.Exists(centerline))

Input FC: True


In [43]:
arcpy.analysis.Near(
    in_features=parcelzoningFLU,
    near_features=centerline,
    search_radius=None,
    location="NO_LOCATION",
    angle="NO_ANGLE",
    method="PLANAR",
    field_names="NEAR_DIST NEAR_DIST",
    distance_unit="Feet")

<Result 'C:\\Users\\Thepr\\Documents\\Consulting\\CityofRedWing\\ArcGISPro\\RedWingOutput.gdb\\ParcelZoningFLU'>

In [44]:
arcpy.management.AlterField(
    in_table=parcelzoningFLU,
    field="NEAR_DIST",
    new_field_name="DIST_CENTERLINE",
    new_field_alias="DIST_CENTERLINE",
    field_type="DOUBLE",
    field_length=20,
    field_is_nullable="NULLABLE")

<Result 'C:\\Users\\Thepr\\Documents\\Consulting\\CityofRedWing\\ArcGISPro\\RedWingOutput.gdb\\ParcelZoningFLU'>

In [45]:
arcpy.management.DeleteField(in_table=parcelzoningFLU,drop_field="NEAR_FID",method="DELETE_FIELDS")

<Result 'C:\\Users\\Thepr\\Documents\\Consulting\\CityofRedWing\\ArcGISPro\\RedWingOutput.gdb\\ParcelZoningFLU'>

In [46]:
arcpy.management.CalculateGeometryAttributes(
    in_features= parcelzoningFLU,
    geometry_property="Acres AREA_GEODESIC",
    length_unit="",
    area_unit="ACRES",
    coordinate_system=None,
    coordinate_format="SAME_AS_INPUT")

<Result 'C:\\Users\\Thepr\\Documents\\Consulting\\CityofRedWing\\ArcGISPro\\RedWingOutput.gdb\\ParcelZoningFLU'>

- **Business Analyst Enrichment**

In [47]:
parcelzoning = os.path.join(outputGDB,'ParcelZoningFLU')
siteBA = os.path.join(outputGDB,'SiteBusinessAnalyst')

In [48]:
arcpy.ba.EnrichLayer(
    parcelzoning,
    siteBA,
    ############ Population Totals ################
    # TOTPOP_CY = Current Year Total Population | POPDENS_CY = Current Year Population Density | TOTPOP_FY = Five-Year Forecast Population
    "populationtotals.TOTPOP_CY;populationtotals.POPDENS_CY;populationtotals.TOTPOP_FY;HistoricalHousing.TOTHU_CY; "\
    #######  Housing Units #####
    # TOTHU_CY = Current Year Total Housing Units | VACANT_CY = Current Year Vacant Units | VACANT_FY = Five-Year Forecast Vacant Units
    # MEDVAL_CY = Current Year Median Home Value  | ACSMEDVAL = ACS Median Home Value
    "vacant.VACANT_CY;vacant.VACANT_FY;housingunittotals.TOTHU_CY;homevalue.MEDVAL_CY;homevalue.ACSMEDVAL; "\
    ################# Retail ##################################
    # INDRTFD_X = Food & Beverage Store Demand (NAICS 445) | IND722_X = Food Services & Drinking Places (NAICS 722)
    # INDRT_X = Total Retail Demand
    "RetailDemandbyNAICS.INDRTFD_X;RetailDemandbyNAICS.IND722_X;RetailDemandbyNAICS.INDRT_X; "\
    ###################  Employment ###############################
    # UNEMPRT_CY = Unemployment Rate | EMP_CY = Total Employment
    # EMPMFG_CY = Manufacturing | EMPAGRI_CY = Agriculture | EMPRETL_CY = Retail | EMPSRVC_CY = Services
    "EmploymentUnemployment.UNEMPRT_CY;EmploymentUnemployment.EMP_CY; "\
    ################### Business Count ############################
    # TOTBUS_CY = Total Businesses Current Year | TOTBUS_FY = Total Businesses Five-Year Forecast
    "businesses.N01_BUS; "\
    "KeyUSFacts.MEDVAL_CY;KeyUSFacts.MEDVAL_FY",
    "",
    0,
    "")

<Result 'C:\\Users\\Thepr\\Documents\\Consulting\\CityofRedWing\\ArcGISPro\\RedWingOutput.gdb\\SiteBusinessAnalyst'>

In [49]:
p = [row.name for row in arcpy.ListFields(siteBA)]
print(p)

['OBJECTID', 'Shape', 'Join_Count', 'TARGET_FID', 'ID', 'Name', 'PIN', 'ZONE_DOC', 'C0CITY', 'C0CITYP', 'C0PYEAR', 'C0CLSFDS', 'C0HSTDDS', 'PLU_1', 'LANDUSEDES', 'REDEV', 'DEV_STATUS', 'ConservationEasement', 'TribalAreas', 'SEHistDistLocal', 'HistDistLocal', 'GreenNetwork', 'PIICOwned', 'RWLocality', 'DIST_WATERMAIN', 'DIST_SEWER', 'DIST_STORMPIPE', 'DIST_CENTERLINE', 'Acres', 'aggregationMethod', 'HasData', 'ORIGINAL_OID', 'sourceCountry', 'apportionmentConfidence', 'populationToPolygonSizeRating', 'populationtotals_TOTPOP_CY', 'populationtotals_POPDENS_CY', 'populationtotals_TOTPOP_FY', 'HistoricalHousing_TOTHU_CY', 'vacant_VACANT_CY', 'vacant_VACANT_FY', 'housingunittotals_TOTHU_CY', 'homevalue_MEDVAL_CY', 'homevalue_ACSMEDVAL', 'RetailDemandbyNAICS_INDRTFD_X', 'RetailDemandbyNAICS_IND722_X', 'RetailDemandbyNAICS_INDRT_X', 'EmploymentUnemployment_UNEMPRT_CY', 'EmploymentUnemployment_EMP_CY', 'businesses_N01_BUS', 'KeyUSFacts_MEDVAL_CY', 'KeyUSFacts_MEDVAL_FY', 'Shape_Length', 'Shap

In [50]:
suitability= os.path.join(workingfolder, 'SiteSuitablity.csv')

arcpy.conversion.ExportTable(in_table=siteBA,out_table=suitability,where_clause="",use_field_alias_as_name="NOT_USE_ALIAS",
                             field_mapping='',sort_field=None)

<Result 'C:\\Users\\Thepr\\Documents\\Consulting\\CityofRedWing\\SiteSuitablity.csv'>

In [51]:
suitability= os.path.join(workingfolder, 'SiteSuitablity.csv')

data = pd.read_csv(suitability)
data.columns

Index(['Join_Count', 'TARGET_FID', 'ID', 'Name', 'PIN', 'ZONE_DOC', 'C0CITY',
       'C0CITYP', 'C0PYEAR', 'C0CLSFDS', 'C0HSTDDS', 'PLU_1', 'LANDUSEDES',
       'REDEV', 'DEV_STATUS', 'ConservationEasement', 'TribalAreas',
       'SEHistDistLocal', 'HistDistLocal', 'GreenNetwork', 'PIICOwned',
       'RWLocality', 'DIST_WATERMAIN', 'DIST_SEWER', 'DIST_STORMPIPE',
       'DIST_CENTERLINE', 'Acres', 'aggregationMethod', 'HasData',
       'ORIGINAL_OID', 'sourceCountry', 'apportionmentConfidence',
       'populationToPolygonSizeRating', 'populationtotals_TOTPOP_CY',
       'populationtotals_POPDENS_CY', 'populationtotals_TOTPOP_FY',
       'HistoricalHousing_TOTHU_CY', 'vacant_VACANT_CY', 'vacant_VACANT_FY',
       'housingunittotals_TOTHU_CY', 'homevalue_MEDVAL_CY',
       'homevalue_ACSMEDVAL', 'RetailDemandbyNAICS_INDRTFD_X',
       'RetailDemandbyNAICS_IND722_X', 'RetailDemandbyNAICS_INDRT_X',
       'EmploymentUnemployment_UNEMPRT_CY', 'EmploymentUnemployment_EMP_CY',
       'busines

## Suitability Model

In [52]:
suitabilityweight ={'Physical':0.25,'Regulatory':0.20,'Market':0.20,'Ownership':0.20,'Strategic':0.15}

physicalweight={'DIST_WATERMAIN_score':0.25,'DIST_SEWER_score':0.25,'DIST_STORMPIPE_score':0.20,
                'DIST_CENTERLINE_score':0.20,'Acres_score':0.10,}

regulatoryweight = {'ZONE_DOC_score':0.25,'PLU_1_score':0.20,'ConservationEasement_score':0.15,'TribalAreas_score':0.10,
                    'HistDistLocal_score':0.10,'SEHistDistLocal_score':0.05,'GreenNetwork_score':0.10}

marketweight = {'TOTPOP_CY_score':0.10,'POPDENS_CY_score':0.10,'TOTPOP_FY_score':0.05,'TOTHU_CY_score':0.05,
                'MEDVAL_CY_score':0.10,'INDRT_X_score':0.05,'INDRTFD_X_score':0.10,'IND722_X_score':0.10,
                'UNEMPRT_CY_score':0.10,'VACANT_CY_score':0.10,'VACANT_FY_score':0.05,'EMP_CY_score':0.05,
                'N01_BUS_score':0.05}

ownershipweight = { 'PIICOwned_score':0.40,'C0HSTDDS_score':0.30,'C0CLSFDS_score': 0.30,}

strategicweight = {'RWLocality_score': 0.70, 'DEVSTATUS_score': 0.30}

#### Categorical Dictionary  (0.0 = least suitable, 1.0 = most suitable)

- Zoning

In [53]:
ZoneDocScores = {
    # --- Commercial / Business ---
    'B3-Central Business':                                  1.0,
    'B2-General Business':                                  0.9,
    'B1-Local Business':                                    0.8,
    # --- Mixed Use ---
    'MC-Mixed Use/Industrial/Office Commercial':            1.0,
    'MCT-Mixed Use Commercial Tourism':                     1.0,
    'RF-Riverfront':                                        0.8,
    # --- Industrial ---
    'I1-Light Industrial':                                  0.8,
    'I2-General Industrial':                                0.7,
    # --- Civic / Institutional ---
    'CI-Civic':                                             0.5,
    # --- Residential (higher density = more developable) ---
    'RM2-Residential Multi-Family Two (+16 units/acre)':    0.7,
    'RM1-Residential Multi-Family One (8-16 units/acre)':   0.6,
    'R2-Residential Two (5-8 units/acre)':                  0.4,
    'R1-Residential One (3.5-5 units/acre)':                0.3,
    'R1-Residential one (3.5-5 units/acre)':                0.3,
    # --- Agriculture / Conservation ---
    'AR-Agriculture Residential':                           0.2,
    'AC-Agriculture Conservation':                          0.1,
    'A-Agriculture':                                        0.1,
    # --- Special ---
    'Prairie Island Indian Community':                      0.0,
    'Split Zoning':                                         0.4,
    'DEFAULT':                                              0.5
}

- Landuse

In [54]:
LandUse = {
    # Fallback when PLU_1 is null or blank
    'Downtown':             1.0,
    'Mixed Use Commercial': 1.0,
    'Community Commercial': 0.9,
    'Vacant':               0.8,
    'Institutional':        0.4,
    'Open Space':           0.1,
    'Agriculture':          0.1,
    'DEFAULT':              0.5,
}

- Future land use

In [55]:
plu = {
    'MUC':     1.0,   # Mixed Use Commercial
    'CC':      0.9,   # Community Commercial
    'BP':      0.8,   # Business Park
    'PSP':     0.4,   # Public / Semi-Public
    'LDR':     0.3,   # Low Density Residential
    'ORF':     0.2,   # Open Recreation / Floodplain
    'AG':      0.1,   # Agriculture
    'DEFAULT': 0.5,
}

In [56]:
colcsfds = {
    'RURAL VACANT LAND':   1.0,   # unimproved — highest acquisition opportunity
    'COMM LAND & BLDGS':   0.7,   # commercial — developable
    'MUNICIPAL PUB-OTHER': 0.6,   # public-owned — aligns with PIICOwned signal
    'AGRICULTURAL':        0.4,   # ag land — conversion possible
    'COLLEGE-PUBLIC':      0.3,   # institutional — limited flexibility
    'K-12 SCH-PUBLIC':     0.2,   # school property — constrained reuse
    'DEFAULT':             0.5,
}

- Conservation, Tribal areas, historical districs, red wing locality, dev status, picc owned

In [57]:
conservationeasement = {
    'Yes': 0.0,   # legally encumbered — not developable
    'No':  1.0}

tribalareas =  {
    'Yes': 0.0,   # outside City jurisdiction
    'No':  1.0}

historicaldist = {
    'Yes': 0.0,   # historic overlay — development constrained
    'No':  1.0}

redwinglocality = {
    'Yes': 1.0,   # within Red Wing boundary — strategically favorable
    'No':  0.2,   # outside boundary — lower strategic priority
}

devstatusscore = {
    # REDEV is entirely blank in this dataset - DEV_STATUS carries the redevelopment signal
    'Infill':      1.0,
    'Edge Growth': 0.7,
    ' ':           0.4,
    '':            0.4,
    'DEFAULT':     0.4,
}

piccowned = {
    'Yes': 1.0,   # public/institutional - lower acquisition complexity
    'No':  0.3,   # privately owned - acquisition harder
}

greennetwork = {
    # All 13 sites = 'Yes' in current dataset
    'Yes': 0.0,   # green network overlay - development constrained
    'No':  1.0,
}

cohstdds = {
    'NON-HOMESTEAD':  0.8,   # not owner-occupied - easier to acquire
    'FULL HOMESTEAD': 0.3,   # owner-occupied - acquisition harder
}

sehistdist = {
    'Yes': 0.0,
    'No':  1.0,
}

In [58]:
fields = [
    'PIN',
    # Physical
    'DIST_WATERMAIN', 'DIST_SEWER', 'DIST_STORMPIPE', 'DIST_CENTERLINE', 'Acres',
    # Regulatory
    'ZONE_DOC', 'PLU_1', 'LANDUSEDES',
    'ConservationEasement', 'TribalAreas', 'SEHistDistLocal', 'HistDistLocal',
    'GreenNetwork', 'REDEV', 'DEV_STATUS',
    # Market
    'populationtotals_TOTPOP_CY', 'populationtotals_POPDENS_CY',
    'populationtotals_TOTPOP_FY', 'housingunittotals_TOTHU_CY',
    'homevalue_MEDVAL_CY', 'homevalue_ACSMEDVAL',
    'RetailDemandbyNAICS_INDRTFD_X', 'RetailDemandbyNAICS_INDRT_X',
    'RetailDemandbyNAICS_IND722_X', 'EmploymentUnemployment_UNEMPRT_CY',
    'KeyUSFacts_MEDVAL_CY', 'KeyUSFacts_MEDVAL_FY',
    # Ownership
    'PIICOwned', 'C0CITY', 'C0CITYP', 'C0CLSFDS', 'C0HSTDDS',
    # Strategic
    'RWLocality', 'C0PYEAR',
]

In [59]:
print("=" * 60)

print('PHYSICAL & INFRASTRUCTURE  (25%)')

print("=" * 60)

PHYSICAL & INFRASTRUCTURE  (25%)


- Watermain

In [60]:
mn, mx = data['DIST_WATERMAIN'].min(), data['DIST_WATERMAIN'].max()
data['DIST_WATERMAIN_score'] = 1 - (data['DIST_WATERMAIN'].fillna(mx) - mn) / (mx - mn) if mx != mn else 0.5

- Sewer

In [61]:
mn, mx = data['DIST_SEWER'].min(), data['DIST_SEWER'].max()
data['DIST_SEWER_score'] = 1 - (data['DIST_SEWER'].fillna(mx) - mn) / (mx - mn) if mx != mn else 0.5

- Stormpipe

In [62]:
mn, mx = data['DIST_STORMPIPE'].min(), data['DIST_STORMPIPE'].max()
data['DIST_STORMPIPE_score'] = 1 - (data['DIST_STORMPIPE'].fillna(mx) - mn) / (mx - mn) if mx != mn else 0.5

- Centerline

In [63]:
mn, mx = data['DIST_CENTERLINE'].min(), data['DIST_CENTERLINE'].max()
data['DIST_CENTERLINE_score'] = 1 - (data['DIST_CENTERLINE'].fillna(mx) - mn) / (mx - mn) if mx != mn else 0.5

- Acres

In [64]:
mn, mx = data['Acres'].min(), data['Acres'].max()
data['Acres_score'] = (data['Acres'].fillna(0) - mn) / (mx - mn) if mx != mn else 0.5

- Physical

In [65]:
data['DIM_Physical'] = (
    data['DIST_WATERMAIN_score']  * physicalweight['DIST_WATERMAIN_score']  +
    data['DIST_SEWER_score']      * physicalweight['DIST_SEWER_score']      +
    data['DIST_STORMPIPE_score']  * physicalweight['DIST_STORMPIPE_score']  +
    data['DIST_CENTERLINE_score'] * physicalweight['DIST_CENTERLINE_score'] +
    data['Acres_score']           * physicalweight['Acres_score']
)

 ## Regulatory

In [66]:
print("=" * 60)

print('REGULATORY  (20%)')

print("=" * 60)

REGULATORY  (20%)


- Zoning

In [67]:
data['ZONE_DOC_score'] = (data['ZONE_DOC'].fillna('DEFAULT').str.strip().map(ZoneDocScores)
                          .fillna(ZoneDocScores['DEFAULT']))

- Future LU

In [68]:
data['PLU_1_score'] = (data['PLU_1'].str.strip().map(plu).fillna(data['LANDUSEDES'].str.strip().map(LandUse)).
                       fillna(plu['DEFAULT']))

- Conservation

In [69]:
data['ConservationEasement_score'] = (data['ConservationEasement'].str.strip().map(conservationeasement).fillna(1.0))

- Tribal 

In [70]:
data['TribalAreas_score'] = (data['TribalAreas'].str.strip().map(tribalareas).fillna(1.0))

- Historical District

In [71]:
data['HistDistLocal_score'] = (data['HistDistLocal'].str.strip().map(historicaldist).fillna(1.0))

- SE Historical District

In [72]:
data['SEHistDistLocal_score'] = (data['SEHistDistLocal'].str.strip().map(sehistdist).fillna(1.0))

- Green Network 

In [73]:
data['GreenNetwork_score'] = (data['GreenNetwork'].str.strip().map(greennetwork).fillna(1.0))

- Dev status

In [74]:
data['DEVSTATUS_score'] = (data['DEV_STATUS'].fillna(' ').str.strip().map(devstatusscore)
                            .fillna(devstatusscore['DEFAULT']))

- Regulatory

In [75]:
data['DIM_Regulatory'] = (
    data['ZONE_DOC_score']             * regulatoryweight['ZONE_DOC_score']             +
    data['PLU_1_score']                * regulatoryweight['PLU_1_score']                +
    data['ConservationEasement_score'] * regulatoryweight['ConservationEasement_score'] +
    data['TribalAreas_score']          * regulatoryweight['TribalAreas_score']          +
    data['HistDistLocal_score']        * regulatoryweight['HistDistLocal_score']        +
    data['SEHistDistLocal_score']      * regulatoryweight['SEHistDistLocal_score']      +
    data['GreenNetwork_score']         * regulatoryweight['GreenNetwork_score']         
)

## Market 

In [76]:
marketfields = [
    'populationtotals_TOTPOP_CY', 'populationtotals_POPDENS_CY',
    'populationtotals_TOTPOP_FY', 'housingunittotals_TOTHU_CY',
    'homevalue_MEDVAL_CY', 'vacant_VACANT_CY', 'vacant_VACANT_FY',
    'RetailDemandbyNAICS_INDRT_X', 'RetailDemandbyNAICS_INDRTFD_X',
    'RetailDemandbyNAICS_IND722_X', 'EmploymentUnemployment_UNEMPRT_CY',
    'EmploymentUnemployment_EMP_CY', 'businesses_N01_BUS']

In [77]:
data['BA_NO_DATA'] = (data[marketfields].fillna(0) == 0).all(axis=1)

no_data_count = data['BA_NO_DATA'].sum()
no_data_count

4

- Total Population

In [78]:
mn, mx = data['populationtotals_TOTPOP_CY'].min(), data['populationtotals_TOTPOP_CY'].max()
data['TOTPOP_CY_score'] = ((data['populationtotals_TOTPOP_CY'].fillna(0) - mn) / (mx - mn) if mx != mn else 0.5)

- Population Density

In [79]:
mn, mx = data['populationtotals_POPDENS_CY'].min(), data['populationtotals_POPDENS_CY'].max()
data['POPDENS_CY_score'] = ((data['populationtotals_POPDENS_CY'].fillna(0) - mn) / (mx - mn) if mx != mn else 0.5)

- Total Population 5-Year Population

In [80]:
mn, mx = data['populationtotals_TOTPOP_FY'].min(), data['populationtotals_TOTPOP_FY'].max()
data['TOTPOP_FY_score'] = (
    (data['populationtotals_TOTPOP_FY'].fillna(0) - mn) / (mx - mn) if mx != mn else 0.5)

- Housing Units

In [81]:
mn, mx = data['housingunittotals_TOTHU_CY'].min(), data['housingunittotals_TOTHU_CY'].max()
data['TOTHU_CY_score'] = ((data['housingunittotals_TOTHU_CY'].fillna(0) - mn) / (mx - mn) if mx != mn else 0.5)

- Median Home Value

In [82]:
log_medval = np.log1p(data['homevalue_MEDVAL_CY'].fillna(0))
mn, mx     = log_medval.min(), log_medval.max()
data['MEDVAL_CY_score'] = (log_medval - mn) / (mx - mn) if mx != mn else 0.5

- Retail Demand 

In [83]:
log_indrt  = np.log1p(data['RetailDemandbyNAICS_INDRT_X'].fillna(0))
mn, mx= log_indrt.min(), log_indrt.max()
data['INDRT_X_score'] = (log_indrt - mn) / (mx - mn) if mx != mn else 0.5

In [84]:
log_ind722 = np.log1p(data['RetailDemandbyNAICS_IND722_X'].fillna(0))
mn, mx= log_ind722.min(), log_ind722.max()
data['IND722_X_score'] = (log_ind722 - mn) / (mx - mn) if mx != mn else 0.5

In [85]:
log_indrtfd = np.log1p(data['RetailDemandbyNAICS_INDRTFD_X'].fillna(0))
mn, mx= log_indrtfd.min(), log_indrtfd.max()
data['INDRTFD_X_score'] = (log_indrtfd - mn) / (mx - mn) if mx != mn else 0.5

- Vacancy Current Year

In [86]:
mn, mx = data['vacant_VACANT_CY'].min(), data['vacant_VACANT_CY'].max()
data['VACANT_CY_score'] = ((data['vacant_VACANT_CY'].fillna(0) - mn) / (mx - mn) if mx != mn else 0.5)

- Vacancy Future Year

In [87]:
mn, mx = data['vacant_VACANT_FY'].min(), data['vacant_VACANT_FY'].max()
data['VACANT_FY_score'] = ((data['vacant_VACANT_FY'].fillna(0) - mn) / (mx - mn) if mx != mn else 0.5)

- Unemployment: lower = stronger labor market = higher suitability (inverted)

In [88]:
mn, mx = data['EmploymentUnemployment_UNEMPRT_CY'].min(), data['EmploymentUnemployment_UNEMPRT_CY'].max()
data['UNEMPRT_CY_score'] = ( 1 - (data['EmploymentUnemployment_UNEMPRT_CY'].fillna(0) - mn) / (mx - mn) 
                            if mx != mn else 0.5)

- Business Count

In [89]:
mn, mx = data['businesses_N01_BUS'].min(), data['businesses_N01_BUS'].max()
data['N01_BUS_score'] = ((data['businesses_N01_BUS'].fillna(0) - mn) / (mx - mn) if mx != mn else 0.5)

- Employment count

In [90]:
mn, mx = data['EmploymentUnemployment_EMP_CY'].min(), data['EmploymentUnemployment_EMP_CY'].max()
data['EMP_CY_score'] = ((data['EmploymentUnemployment_EMP_CY'].fillna(0) - mn) / (mx - mn) if mx != mn else 0.5)

- Market

In [91]:
data['DIM_Market'] = (
    data['TOTPOP_CY_score']  * marketweight['TOTPOP_CY_score']  +
    data['POPDENS_CY_score'] * marketweight['POPDENS_CY_score'] +
    data['TOTPOP_FY_score']  * marketweight['TOTPOP_FY_score']  +
    data['TOTHU_CY_score']   * marketweight['TOTHU_CY_score']   +
    data['VACANT_CY_score']  * marketweight['VACANT_CY_score']  +
    data['VACANT_FY_score']  * marketweight['VACANT_FY_score']  +
    data['MEDVAL_CY_score']  * marketweight['MEDVAL_CY_score']  +
    data['INDRT_X_score']    * marketweight['INDRT_X_score']    +
    data['INDRTFD_X_score']  * marketweight['INDRTFD_X_score']  +
    data['IND722_X_score']   * marketweight['IND722_X_score']   +
    data['UNEMPRT_CY_score'] * marketweight['UNEMPRT_CY_score'] +
    data['EMP_CY_score']     * marketweight['EMP_CY_score']     +
    data['N01_BUS_score']    * marketweight['N01_BUS_score']
)

### Ownership

- PIIC Owned

In [92]:
data['PIICOwned_score'] = (data['PIICOwned'].str.strip().map(piccowned).fillna(piccowned['No']))

- COHSTDDS

In [93]:
data['C0HSTDDS_score'] = (data['C0HSTDDS'].str.strip().map(cohstdds).fillna(cohstdds['NON-HOMESTEAD']))

- Colcsfds

In [94]:
data['C0CLSFDS_score'] = (data['C0CLSFDS'].fillna('DEFAULT').str.strip().map(colcsfds).fillna(colcsfds['DEFAULT']))

- Ownership

In [95]:
data['DIM_Ownership'] = (
    data['PIICOwned_score'] * ownershipweight['PIICOwned_score'] +
    data['C0HSTDDS_score']  * ownershipweight['C0HSTDDS_score']  +
    data['C0CLSFDS_score']  * ownershipweight['C0CLSFDS_score']
)

### Strategic

In [96]:
data['RWLocality_score'] = (data['RWLocality'].str.strip().map(redwinglocality).fillna(redwinglocality['No']))

In [98]:
data['DIM_Strategic'] = (
    data['RWLocality_score'] * strategicweight['RWLocality_score'] +
    data['DEVSTATUS_score'] * strategicweight['DEVSTATUS_score']
)

### SRI SCORE

In [99]:
data['SRI_Score'] = (
    data['DIM_Physical']   * suitabilityweight['Physical']   +
    data['DIM_Regulatory'] * suitabilityweight['Regulatory'] +
    data['DIM_Market']     * suitabilityweight['Market']     +
    data['DIM_Ownership']  * suitabilityweight['Ownership']  +
    data['DIM_Strategic']  * suitabilityweight['Strategic']
) * 100

### TIER CLASSIFICATION

In [100]:
data['SRI_Tier'] = pd.cut(
    data['SRI_Score'],
    bins=[0, 40, 55, 70, 85, 100],
    labels=[
        'Tier 5 - Low',
        'Tier 4 - Below Average',
        'Tier 3 - Moderate',
        'Tier 2 - High',
        'Tier 1 - Highest',
    ],
    include_lowest=True
).astype(str)

- Rank

In [101]:
data['SRI_Rank'] = data['SRI_Score'].rank(ascending=False, method='min').astype(int)
data.head(3)

,Join_Count,TARGET_FID,ID,Name,PIN,ZONE_DOC,C0CITY,C0CITYP,C0PYEAR,C0CLSFDS,...,DIM_Market,PIICOwned_score,C0HSTDDS_score,C0CLSFDS_score,DIM_Ownership,RWLocality_score,DIM_Strategic,SRI_Score,SRI_Tier,SRI_Rank
0,10,1,A,Bridge Landing,550052540,B3-Central Business,RED WING,RED WING,2018.0,MUNICIPAL PUB-OTHER,...,0.101754,0.3,0.8,0.6,0.54,0.2,0.26,56.235088,Tier 3 - Moderate,5
1,63,2,B,Downtown Riverfront,550054500,B3-Central Business,,RED WING,2018.0,COMM LAND & BLDGS,...,0.590293,1.0,0.8,0.7,0.85,0.2,0.26,67.892676,Tier 3 - Moderate,3
2,109,3,C,Old West Main,555800381,MCT-Mixed Use Commercial Tourism,,MINNEAPOLIS,2018.0,COMM LAND & BLDGS,...,0.686900,0.3,0.8,0.7,0.57,1.0,0.82,77.008687,Tier 2 - High,1


- Write Score Back to Feature Class

In [102]:
siteBA = os.path.join(outputGDB,'SiteBusinessAnalyst')
siteBAFinal = os.path.join(outputGDB,'SiteBusinessAnalystFinal')

In [103]:
if arcpy.Exists(siteBAFinal):
    arcpy.Delete_management(siteBAFinal)
arcpy.CopyFeatures_management(siteBA, siteBAFinal)
 
score_fields = {
    'DIM_Physical':   'DOUBLE',
    'DIM_Regulatory': 'DOUBLE',
    'DIM_Market':     'DOUBLE',
    'DIM_Ownership':  'DOUBLE',
    'DIM_Strategic':  'DOUBLE',
    'SRI_Score':      'DOUBLE',
    'SRI_Rank':       'LONG',
    'SRI_Tier':       'TEXT',
    'BA_NO_DATA':     'SHORT',
}

In [104]:
existing = {f.name for f in arcpy.ListFields(siteBAFinal)}
for field, ftype in score_fields.items():
    if field not in existing:
        if ftype == 'TEXT':
            arcpy.management.AddField(siteBAFinal, field, ftype, field_length=50)
        else:
            arcpy.management.AddField(siteBAFinal, field, ftype)

In [105]:
writefields = ['ID', 'Name','DIM_Physical', 'DIM_Regulatory', 'DIM_Market','DIM_Ownership', 'DIM_Strategic', 'SRI_Score',
               'SRI_Rank', 'SRI_Tier', 'BA_NO_DATA']

updated = 0
with arcpy.da.UpdateCursor(siteBAFinal, writefields) as cursor:
    for row in cursor:
        match = data[data['ID'] == row[0]]
        if not match.empty:
            row[1] = round(float(match['DIM_Physical'].values[0]),   4)
            row[2] = round(float(match['DIM_Regulatory'].values[0]), 4)
            row[3] = round(float(match['DIM_Market'].values[0]),     4)
            row[4] = round(float(match['DIM_Ownership'].values[0]),  4)
            row[5] = round(float(match['DIM_Strategic'].values[0]),  4)
            row[6] = round(float(match['SRI_Score'].values[0]),      2)
            row[7] = int(match['SRI_Rank'].values[0])
            row[8] = str(match['SRI_Tier'].values[0])
            row[9] = int(bool(match['BA_NO_DATA'].values[0]))
            cursor.updateRow(row)
            updated += 1
updated

13

### Write output to Excel

In [106]:
outputexcel  = os.path.join(workingfolder, 'SRI_Indicator_Workbook.xlsx')


In [113]:
print(f"\nWriting Excel workbook to {outputexcel}")

os.makedirs(os.path.dirname(outputexcel), exist_ok=True)
 
with pd.ExcelWriter(outputexcel, engine='openpyxl') as writer:
 
    # Summary
    data[['Name','ID', 'PIN',
          'DIM_Physical', 'DIM_Regulatory', 'DIM_Market',
          'DIM_Ownership', 'DIM_Strategic',
          'SRI_Score', 'SRI_Rank', 'SRI_Tier', 'BA_NO_DATA']
    ].sort_values('SRI_Rank').to_excel(writer, sheet_name='Summary', index=False)
 
    # Physical
    data[['ID', 'PIN',
          'DIST_WATERMAIN',  'DIST_WATERMAIN_score',
          'DIST_SEWER',      'DIST_SEWER_score',
          'DIST_STORMPIPE',  'DIST_STORMPIPE_score',
          'DIST_CENTERLINE', 'DIST_CENTERLINE_score',
          'Acres',           'Acres_score',
          'DIM_Physical']
    ].to_excel(writer, sheet_name='Physical', index=False)
 
    # Regulatory
    data[['ID', 'PIN',
          'ZONE_DOC',             'ZONE_DOC_score',
          'PLU_1',                'PLU_1_score',
          'ConservationEasement', 'ConservationEasement_score',
          'TribalAreas',          'TribalAreas_score',
          'HistDistLocal',        'HistDistLocal_score',
          'SEHistDistLocal',      'SEHistDistLocal_score',
          'GreenNetwork',         'GreenNetwork_score',
          'DEV_STATUS',           'DEVSTATUS_score',
          'DIM_Regulatory']
    ].to_excel(writer, sheet_name='Regulatory', index=False)
 
    # Market
    data[['ID', 'PIN', 'BA_NO_DATA',
          'populationtotals_TOTPOP_CY',        'TOTPOP_CY_score',
          'populationtotals_POPDENS_CY',       'POPDENS_CY_score',
          'populationtotals_TOTPOP_FY',        'TOTPOP_FY_score',
          'housingunittotals_TOTHU_CY',        'TOTHU_CY_score',
          'homevalue_MEDVAL_CY',               'MEDVAL_CY_score',
          'RetailDemandbyNAICS_INDRT_X',       'INDRT_X_score',
          'RetailDemandbyNAICS_INDRTFD_X',     'INDRTFD_X_score',
          'RetailDemandbyNAICS_IND722_X',      'IND722_X_score',
          'EmploymentUnemployment_UNEMPRT_CY', 'UNEMPRT_CY_score',
          'DIM_Market']
    ].to_excel(writer, sheet_name='Market', index=False)
 
    # Ownership
    data[['ID', 'PIN',
          'PIICOwned', 'PIICOwned_score',
          'C0HSTDDS',  'C0HSTDDS_score',
          'C0CLSFDS',  'C0CLSFDS_score',
          'DIM_Ownership']
    ].to_excel(writer, sheet_name='Ownership', index=False)
 
    # Strategic
    data[['ID', 'PIN',
          'RWLocality', 'RWLocality_score',
          'DEV_STATUS', 'DEVSTATUS_score',
          'C0PYEAR',
          'DIM_Strategic']
    ].to_excel(writer, sheet_name='Strategic', index=False)
    
print(f"\nWorkbook written.")
print("\nSRI model complete.")


Writing Excel workbook to C:\Users\Thepr\Documents\Consulting\CityofRedWing\SRI_Indicator_Workbook.xlsx

Workbook written.

SRI model complete.
